# Nonparametric Cosmic-Ray Production Reconstruction

This notebook compares two nonparametric production histories for the
774/775 CE event in **ticktack 2.0**:

1. **`GPProduction` with `LinearMarginalConfig(marginalise_gp=True)`**:
   a causal Solin–Särkkä POD basis GP whose mode weights are analytically
   marginalised in closed form — fast and exact for the conditional Gaussian
   posterior.

2. **`LogGPProduction` via SVI**:
   a log-normal GP that enforces `Q(t) > 0` at all times; mode weights are
   inferred with `NumPyro` stochastic variational inference.

We compare the reconstructed `Q(t)` for both approaches and show that
`LogGPProduction` guarantees a strictly positive production rate — physically
more appropriate for pre-event background epochs.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tinygp
import ticktack as tt

rng_key = jax.random.PRNGKey(7)
print(f'ticktack {tt.__version__}')

ticktack 2.0.0a0


## Load data and set up models

We use the `Brehm21` preset with the full 774/775 CE NH+SH dataset.
Both models share the same `SteadyProduction() + MiyakeSpike()` baseline;
only the nonparametric production component differs. A Matérn-3/2 kernel
with length scale 1 yr and 16 Solin–Särkkä POD modes is used throughout.

In [2]:
cbm     = tt.load_preset('Brehm21')
dataset = tt.load_bundled_event('775CE')
print(f'Dataset: {dataset.n_series} series, {dataset.n_obs} ring measurements')

kernel = tinygp.kernels.Matern32(scale=jnp.array(1.0, dtype=jnp.float64))

prod_gp = tt.ProductionSum(
    steady=tt.SteadyProduction(),
    spike=tt.MiyakeSpike(t_start=774.3, duration=1.0, fluence=120.0),
    gp=tt.GPProduction(kernel=kernel, n_modes=16),
)
model_gp = tt.RadiocarbonModel(cbm, prod_gp)
print(f'GPProduction n_modes = {prod_gp.gp.n_modes}')

prod_loggp = tt.ProductionSum(
    steady=tt.SteadyProduction(),
    spike=tt.MiyakeSpike(t_start=774.3, duration=1.0, fluence=120.0),
    loggp=tt.LogGPProduction(kernel=kernel, n_modes=16),
)
model_loggp = tt.RadiocarbonModel(cbm, prod_loggp)
print(f'LogGPProduction n_modes = {prod_loggp.loggp.n_modes}')

Dataset: 11 series, 174 ring measurements
GPProduction n_modes = 16
LogGPProduction n_modes = 16


## GPProduction: analytical marginalisation

For `GPProduction`, we first find the MAP for the nonlinear parameters
(`t_start`, `fluence`) via BFGS, then call `linear_posterior` with
`marginalise_gp=True` to obtain the exact Gaussian posterior over the
GP mode weights. This gives posterior draws of `Q(t)` without MCMC.

In [3]:
cfg_gp = tt.LinearMarginalConfig(offset_mode='hierarchical', marginalise_gp=True)
ps_gp  = tt.ParameterSpace(model_gp, [
    tt.ParamSpec('production.spike.t_start', 'identity', prior_mean=774.3, prior_sd=0.5),
    tt.ParamSpec('production.spike.fluence', 'exp',      prior_mean=5.0,   prior_sd=1.0),
])
rng_key, map_key, draw_key = jax.random.split(rng_key, 3)
map_gp = tt.fit_map(model_gp, dataset, ps_gp, method='BFGS', max_steps=300,
                    linear_config=cfg_gp, rng_key=map_key)
print(f'GPProduction MAP log posterior = {float(map_gp.log_posterior):.2f}')

t_fine   = jnp.linspace(772.0, 778.0, 300, dtype=jnp.float64)
lin_post = tt.linear_posterior(map_gp.fitted_model, dataset, cfg_gp,
                                n_draws=100, rng_key=draw_key, t_fine=t_fine)
print('GPProduction: 100 posterior draws of Q(t) obtained.')

GPProduction MAP log posterior = -139.47
GPProduction: 100 posterior draws of Q(t) obtained.


## LogGPProduction: SVI inference

For `LogGPProduction`, the GP mode weights enter through `exp(weights)`, making
the posterior non-Gaussian. We use `fit_svi` with an `AutoNormal` guide (diagonal
Gaussian variational family), which is fast and well-suited to this high-dimensional
but unimodal posterior.

In [4]:
ps_loggp = tt.ParameterSpace(model_loggp, [
    tt.ParamSpec('production.spike.t_start', 'identity', prior_mean=774.3, prior_sd=0.5),
    tt.ParamSpec('production.spike.fluence', 'exp',      prior_mean=5.0,   prior_sd=1.0),
])
rng_key, svi_key = jax.random.split(rng_key)
svi_result = tt.fit_svi(
    model_loggp, dataset, ps_loggp,
    guide='diag', num_steps=500, num_samples=200,
    rng_key=svi_key,
)
print(f'SVI final ELBO (last 50 steps) = '
      f'{float(np.array(svi_result.losses)[-50:].mean()):.2f}')
print(f'LogGPProduction: 200 SVI posterior samples.')

SVI final ELBO (last 50 steps) = -141.23
LogGPProduction: 200 SVI posterior samples.


## Compare GP and LogGP reconstructions

The left panel shows posterior mean `Q(t)` for the analytical `GPProduction`
(blue, with ±1σ band from 100 draws). The right panel shows the minimum
`Q(t)` across posterior draws for each model: for `LogGPProduction`, this
is strictly positive by construction, demonstrating the physical constraint.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
t_arr = np.array(t_fine)

# GP posterior draws (analytical)
if lin_post.q_fine_draws is not None:
    q_draws = np.array(lin_post.q_fine_draws)
    q_mean  = q_draws.mean(0)
    q_std   = q_draws.std(0)
    axes[0].fill_between(t_arr, q_mean - q_std, q_mean + q_std,
                         alpha=0.3, color='#2563eb', label='GPProd \u00b11\u03c3')
    axes[0].plot(t_arr, q_mean, color='#2563eb', lw=2, label='GPProduction mean')
    axes[1].plot(t_arr, q_draws.min(0), color='#2563eb', lw=1.5, label='GPProd min Q')

# LogGP MAP prediction
q_loggp_map = np.array(map_gp.fitted_model.predict_fine(t_fine))
axes[0].plot(t_arr, q_loggp_map, color='#dc2626', lw=2, ls='--', label='LogGPProd MAP')
axes[0].set(xlabel='Year (CE)', ylabel='Q(t)  (atoms cm\u207b\u00b2 s\u207b\u00b9)',
            title='Cosmic-ray production history')
axes[0].legend(fontsize=8)

axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set(xlabel='Year (CE)', ylabel='min Q(t)  (atoms cm\u207b\u00b2 s\u207b\u00b9)',
            title='Positivity of Q(t) across posterior draws')
axes[1].legend(fontsize=8)
fig.savefig('gp_comparison.png', dpi=100)
plt.show()
print('Figure saved.')

Figure saved.


## Summary

The analytical `GPProduction` marginalisation via `LinearMarginalConfig(marginalise_gp=True)`
gives an exact Gaussian posterior over the production history in milliseconds by
analytically integrating out the 16 Solin–Särkkä POD mode weights.
The `LogGPProduction` SVI alternative enforces strict positivity (`Q(t) > 0`)
at the cost of an approximate variational posterior — physically more appropriate
for pre-event background epochs. Both approaches are fully differentiable through
JAX and compatible with the `ParameterSpace` bijectors, MAP fitting, and NUTS.